# Silver Linings simulation surrogate model estimation

This notebook builds a surrogate model for the macroeconomic impacts of improvements in biological aging.

To run this notebooks, use the `dynviz-dev` virtual environment, which can be build using the `environment.yml` file located in the same directory as this notebook.

## 0. Import packages

In [1]:
# imports
import numpy as np
import os
import pickle
import pandas as pd
import scipy.interpolate as si
import copy
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
import tensorflow_decision_forests as tfdf
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn import preprocessing


## 1. Fit and test an interpolation model

### 1.1. Fit the linear interpolant

In [3]:
# Function to select data
def select_data(df, output_variables):
    if "NPV" in output_variables:
        input_variables = [
            "age_effect",
            "initial_effect",
            "final_effect",
            "mort_effect",
            "prod_effect",
            "fert_effect",
            "discount_rate",
        ]
    elif "pop_diffs_2050" in output_variables:
        input_variables = [
            "age_effect",
            "initial_effect",
            "final_effect",
            "mort_effect",
            "fert_effect",
        ]
    elif "avg_diff" in output_variables:
        input_variables = [
            "age_effect",
            "initial_effect",
            "final_effect",
            "mort_effect",
            "prod_effect",
            "fert_effect",
        ]
    else:
        print("No valid output variable selected")
    # Depending on the sample, there may be duplicates because, e.g.,
    # the discount rate doesn't affect the population changes
    df.drop_duplicates(subset=input_variables, inplace=True)
    X = df[input_variables].values
    y = df[output_variables].values
    print("Size of X", X.shape)
    print("Size of y", y.shape)

    return X, y, df[input_variables + output_variables].copy()

In [33]:
# Create training data for RBF
# read in data to train the model
df = pd.read_csv("/Users/jason.debacker/repos/OG-Lifespan/DynamicViz/all_sources_together.csv")
print("Size of data: ", df.shape)

# Determine which variables will interpolate over
interpolant_dict = {}
for ov in ["NPV", "avg_diff", "pop_diffs_2050"]:
    output_variables = [ov]
    print("Output variable: ", output_variables)

    # Split the X and y variables
    X, y, df2 = select_data(df, output_variables)

    # Estimate interpolation function using the training data
    interp_f = si.LinearNDInterpolator(
        X, y # might only be able to do linear for more than 2D
    )
    # put interpolation functions into a dictionary
    interpolant_dict[ov] = interp_f
    # save the interpolant
    with open(f"interpolant_{ov}.pkl", "wb") as f:
        pickle.dump(interp_f, f)
print("Interpolants saved to disk")
print("Interpolant dict: ", interpolant_dict.keys())


Size of data:  (2832, 14)
Output variable:  ['NPV']
Size of X (2832, 7)
Size of y (2832, 1)
Output variable:  ['avg_diff']
Size of X (472, 6)
Size of y (472, 1)
Output variable:  ['pop_diffs_2050']
Size of X (241, 5)
Size of y (241, 1)
Interpolants saved to disk
Interpolant dict:  dict_keys(['NPV', 'avg_diff', 'pop_diffs_2050'])


### 1.2. Test how closely the interpolant fits the training data

In [ ]:
# Read interpolants from disk
interpolant_dict = {}
for ov in ["NPV", "avg_diff", "pop_diffs_2050"]:
    with open(f"interpolant_{ov}.pkl", "rb") as f:
        interp_f = pickle.load(f)
    # put interpolation functions into a dictionary
    interpolant_dict[ov] = interp_f
print("Interpolant dict: ", interpolant_dict.keys())

# Now get predicted values from RBF on all the data and compare
df = pd.read_csv("/Users/jason.debacker/repos/OG-Lifespan/DynamicViz/all_sources_together.csv")
for k, v in interpolant_dict.items():
    print("Output variable: ", k)
    X, y, df2 = select_data(df, [k])
    pred_vals = v(X)
    df2[k + "_pred"] = pred_vals[:, 0]
    df2[k + "_diff"] = (
        df2[k] - df2[k + "_pred"]
    )
    print("For {} the mean abs diff is ".format(k), np.absolute(df2[k + "_diff"]).mean())
    print("For {} the max abs diff is".format(k), df2[k + "_diff"].max())
    print("For {} the min abs diff is".format(k), df2[k + "_diff"].min())

Output variable:  ['NPV']
Output variable:  ['avg_diff']
Output variable:  ['pop_diffs_2050']
Interpolant dict:  dict_keys(['NPV', 'avg_diff', 'pop_diffs_2050'])
Output variable:  NPV
Size of X (2832, 7)
Size of y (2832, 1)
For NPV the mean abs diff is  3.1205660501054447e-15
For NPV the max abs diff is 1.7763568394002505e-13
For NPV the min abs diff is -6.288303211476887e-13
Output variable:  avg_diff
Size of X (472, 6)
Size of y (472, 1)
For avg_diff the mean abs diff is  6.015791747274779e-14
For avg_diff the max abs diff is 9.094947017729282e-13
For avg_diff the min abs diff is -4.092726157978177e-12
Output variable:  pop_diffs_2050
Size of X (241, 5)
Size of y (241, 1)
For pop_diffs_2050 the mean abs diff is  4.1301348278834744e-16
For pop_diffs_2050 the max abs diff is 7.105427357601002e-14
For pop_diffs_2050 the min abs diff is -3.552713678800501e-15


### 1.3. Create images to show how the interpolant looks for each output variable and varying each input variable

#### 1.3.1. Look at the interpolant fit for brain aging
The default values for our brain aging simulations are the following:

|    Parameter   | Value |  Min  |  Max  |
| :------------: | :---: | :---: | :---: |
|     age_effect |    50 |    40 |    65 |
| initial_effect |    10 |     0 |    20 |
|   final_effect |    10 |     0 |    20 |
|    mort_effect |  0.02 |  0.00 |  5.00 |
|    prod_effect |  0.07 |  0.00 |  5.00 |
|    fert_effect |  0.00 |  0.00 |  1.50 |

In [ ]:
n_obs = 500
counterfactual = "Brain aging"

for ov in ["NPV", "avg_diff", "pop_diffs_2050"]:
    output_variables = [ov]
    interp_f = interpolant_dict[ov]
    print("Output variable: ", output_variables)
    if "NPV" in output_variables:
        input_vars = {
            "age_effect": [40, 40, 65], # first element is the default value, second is the min, third is the max
            "initial_effect": [10, 0, 20],
            "final_effect": [10, 0, 20],
            "mort_effect": [0.2, 0.0, 5.0],
            "prod_effect": [0.7, 0.0, 5.0],
            "fert_effect": [0.0, 0.0, 1.5],
            "discount_rate": [0.02, 0.01, 0.07]
        }
    elif "avg_diff" in output_variables:
        input_vars = {
            "age_effect": [40, 40, 65], # first element is the default value, second is the min, third is the max
            "initial_effect": [10, 0, 20],
            "final_effect": [10, 0, 20],
            "mort_effect": [0.2, 0.0, 5.0],
            "prod_effect": [0.7, 0.0, 5.0],
            "fert_effect": [0.0, 0.0, 1.5]
        }
    else:
        input_vars = {
            "age_effect": [40, 40, 65], # first element is the default value, second is the min, third is the max
            "initial_effect": [10, 0, 20],
            "final_effect": [10, 0, 20],
            "mort_effect": [0.2, 0.0, 5.0],
            "fert_effect": [0.0, 0.0, 1.5]
        }
    y_defaults = {"NPV": 19.07, "avg_diff": 451.53, "pop_diffs_2050": 0.516}
    y_default = y_defaults[output_variables[0]]

    input_var_names = []
    # Create array of default values for the input variables (these are values in the book)
    X_default = []
    for k, v in input_vars.items():
        X_default.append(v[0])
    # Convert X_default to a numpy array
    X_default = np.array(X_default)

    # Split the X and y variables
    X, y, df2 = select_data(df, output_variables)

    # Now create ranges of the input variables to plot
    length = len(input_vars)
    for i, output_variable in enumerate(output_variables):
        j = 0
        for k, v in input_vars.items():
            # Create the interpolated data over which to plot
            X = np.tile(X_default.reshape((1, length)), (n_obs, 1))
            X[:, j] = np.linspace(v[1], v[2], n_obs)

            # Get predictions
            pred_val= interp_f(X)
            # Plot how the interpolated predictions vary with the input var
            plt.plot(X[:, j], pred_val[:, i], label="interpolated")
            plt.scatter(X_default[j], y_default, label="data", s=10, c="red")
            plt.legend()
            plt.xlabel(k)
            plt.ylabel(output_variable)
            plt.title(f"{counterfactual}: varying {k}")
            plt.savefig(
                f"./images/brain/{output_variable}_{k}.png"
            )
            plt.close()
            j += 1

Output variable:  ['NPV']
Size of X (241, 7)
Size of y (241, 1)
X shape:  (500, 7) 7
X shape:  (500, 7) 7
X shape:  (500, 7) 7
X shape:  (500, 7) 7
X shape:  (500, 7) 7
X shape:  (500, 7) 7
X shape:  (500, 7) 7
Output variable:  ['avg_diff']
Size of X (241, 6)
Size of y (241, 1)
X shape:  (500, 6) 6
X shape:  (500, 6) 6
X shape:  (500, 6) 6
X shape:  (500, 6) 6
X shape:  (500, 6) 6
X shape:  (500, 6) 6
Output variable:  ['pop_diffs_2050']
Size of X (241, 5)
Size of y (241, 1)
X shape:  (500, 5) 5
X shape:  (500, 5) 5
X shape:  (500, 5) 5
X shape:  (500, 5) 5
X shape:  (500, 5) 5


# Use Interpolation Function to Create more data

In [10]:
# Use interpolant to create more data
df = pd.read_csv("/Users/jason.debacker/repos/OG-Lifespan/DynamicViz/all_sources_together.csv")
age_effect_list = [45, 50, 55, 57, 60, 62]  # [40, 50, 65]
initial_effect_list = [1, 4, 8, 10, 12, 15]  # [0, 10, 20]
final_effect_list = [1, 4, 8, 10, 12, 15]  # [0, 5, 10, 20]
mort_effect_list = [0.25, 0.75, 1.5, 3, 4]
prod_effect_list = [0.25, 0.75, 1.5, 3, 4]
fert_effect_list = [0.1, 0.25, 0.4, 0.7, 1.2]
int_rate_list = [0.01, 0.02, 0.04, 0.05, 0.06]
# create a dataframe with combinations of the above lists
from itertools import product
combinations = list(product(
    age_effect_list,
    initial_effect_list,
    final_effect_list,
    mort_effect_list,
    prod_effect_list,
    fert_effect_list,
    int_rate_list

))
# Create a DataFrame from the combinations
new_data = pd.DataFrame(combinations, columns=[
    "age_effect",
    "initial_effect",
    "final_effect",
    "mort_effect",
    "prod_effect",
    "fert_effect",
    "discount_rate"
])
# predict the y values for the new data using the linear interpolator
new_X = new_data[
    [
        "age_effect",
        "initial_effect",
        "final_effect",
        "mort_effect",
        "prod_effect",
        "fert_effect",
        "discount_rate"
    ]
].values
# Make new predictions and add to the new_data dataframe
for k, v in interpolant_dict.items():
    if k == "NPV":
        new_data[k] = v(new_X)
    elif k == "avg_diff":
        X_slim = new_X[:, :-1]
        new_data[k] = v(X_slim)
    elif k == "pop_diffs_2050":
        # Want to drop the last and 3rd to last columns
        X_slim = np.delete(new_X, [4, 6], axis=1)
        new_data[k] = v(X_slim)

# append new_data to df
new_df = pd.concat([df, new_data], ignore_index=True)
new_df.to_csv("augmented_data_with_linear_predictions.csv", index=False)
print("Augmented data saved to augmented_data_with_linear_predictions.csv")



Augmented data saved to augmented_data_with_linear_predictions.csv
